# SigLIP2-SO400M Macro Router — Inference & Evaluation
Held-out evaluation on both test datasets + temperature scaling + per-class thresholds

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Check checkpoint

In [ ]:
import torch

path = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/SigLIP2_SO400M_best.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
print(f"F1: {ckpt['best_f1']:.4f}, Epoch: {ckpt['epoch']}")
if "class_thresholds" in ckpt:
    print(f"Thresholds: {ckpt['class_thresholds']}")
else:
    print("No per-class thresholds in checkpoint")

### Held-out evaluation

In [ ]:
"""
============================================================
   HELD-OUT EVALUATION — 3 Macro Router Models
   Tests on BOTH held-out datasets (never seen during training)
============================================================
Usage (Colab):
    !pip install --upgrade transformers>=4.56.0 timm>=1.0.20 --quiet
    Then run this script.
"""

import os
import cv2
import json
import time
import numpy as np
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score
)
from tqdm import tqdm
import gc

# ==============================================================
# CONFIG
# ==============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# TWO held-out test datasets (NEVER used in training)
EVAL_DATASETS = {
    "classified_images_13_14_test": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Testing_Dataset/classified_images_13_14_test",
    "Careys_labelled_data": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Testing_Dataset/Careys_labelled_data",
}

# Trained model checkpoints
MODEL_DIRS = [
    #"/content/drive/Shareddrives/Garment Type/model_comparison",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models",
]

# Where to save evaluation results
OUTPUT_DIR = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/eval_results"

BATCH_SIZE = 32
NUM_WORKERS = 4

# --- Preprocessing (must match training exactly) ---
APPLY_PREPROCESSING = True
BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING — handles ALL folder name variants
# ==============================================================

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# Exhaustive folder-name → macro-class mapping (covers both datasets)
FOLDER_TO_MACRO = {
    # Upper — Title case (Careys_labelled_data)
    "Blazer": "Upper",
    "Jumpers": "Upper",
    "Shirt": "Upper",
    "T-shirt": "Upper",
    "Dresses": "Upper",
    "Fleece": "Upper",
    # Upper — lowercase (classified_images_13_14_test)
    "blazer": "Upper",
    "jumpers": "Upper",
    "shirt": "Upper",
    "t-shirt": "Upper",
    "dresses": "Upper",
    "fleece": "Upper",

    # Lower — Title case
    "Jeans": "Lower",
    "Trousers": "Lower",
    "Jogging_Bottoms": "Lower",
    "Skirts": "Lower",
    # Lower — lowercase
    "jeans": "Lower",
    "trousers": "Lower",
    "jogging_bottoms": "Lower",
    "skirts": "Lower",

    # Underwear — Title case
    "Bra": "Underwear",
    "Knicker": "Underwear",
    # Underwear — lowercase + typo
    "bra": "Underwear",
    "knicker": "Underwear",
    "kincker": "Underwear",   # typo in dataset 1

    # Other (Towel)
    "Towel": "Other",
    "towel": "Other",
}

# Folders to SKIP (not garment classes)
SKIP_FOLDERS = {"13_14th_test", "unknown", "to_test", "SKIP"}

# ==============================================================
# MODEL CONFIGS (only 3 models — no EfficientNetV2)
# ==============================================================

MODEL_CONFIGS = {
    "SigLIP2_SO400M": {
        "model_id": "google/siglip2-so400m-patch14-384",
        "type": "siglip2",
        "img_size": 384,
    },
    "DINOv3_ViT_Base": {
        "model_id": "vit_base_patch16_dinov3.lvd1689m",
        "type": "timm",
        "img_size": 384,
    },
}

# ==============================================================
# PREPROCESSING (exact copy from training)
# ==============================================================

def preprocess_crop(img_np, bg_color=BACKGROUND_COLOR, margin_ratio=MARGIN_RATIO,
                    black_thresh=BLACK_THRESH):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > black_thresh, 255, 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)

    gray_bg = np.full_like(img_np, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)

    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), bg_color, dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]

    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), bg_color, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped

    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))


# ==============================================================
# DATASET
# ==============================================================

def collect_samples(root):
    """Collect (path, label, original_class) tuples from folder structure."""
    samples = []
    skipped_folders = []
    unmapped_folders = []

    if not os.path.isdir(root):
        print(f"  ❌ Directory not found: {root}")
        return samples

    for folder_name in sorted(os.listdir(root)):
        folder_path = os.path.join(root, folder_name)
        if not os.path.isdir(folder_path):
            continue

        # Skip known non-class folders
        if folder_name in SKIP_FOLDERS:
            skipped_folders.append(folder_name)
            continue

        # Map folder to macro class
        macro_class = FOLDER_TO_MACRO.get(folder_name)
        if macro_class is None:
            unmapped_folders.append(folder_name)
            continue

        label = CLASS_TO_IDX[macro_class]
        count = 0
        for f in os.listdir(folder_path):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(folder_path, f), label, folder_name))
                count += 1
        print(f"    {folder_name:20s} → {macro_class:10s} ({count} images)")

    if skipped_folders:
        print(f"    ⏭️  Skipped folders: {', '.join(skipped_folders)}")
    if unmapped_folders:
        print(f"    ⚠️  Unmapped folders: {', '.join(unmapped_folders)}")

    return samples


class EvalDataset(Dataset):
    def __init__(self, samples, transform, normalize_fn=None, apply_preprocess=True):
        self.samples = samples
        self.transform = transform
        self.normalize_fn = normalize_fn
        self.apply_preprocess = apply_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, orig_cls = self.samples[idx]

        if self.apply_preprocess:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img = Image.new("RGB", (384, 384), (128, 128, 128))
            else:
                img = preprocess_crop(img_bgr)
        else:
            img = Image.open(path).convert("RGB")

        pixel_values = self.transform(img)
        if self.normalize_fn:
            pixel_values = self.normalize_fn(pixel_values)

        return pixel_values, label, idx


# ==============================================================
# CLASSIFIER HEAD (must match training)
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        return self.fc(self.dropout(x))


# ==============================================================
# MODEL LOADING
# ==============================================================

def load_backbone(config):
    """Load vision backbone architecture (no weights yet)."""
    model_type = config["type"]
    model_id = config["model_id"]

    if model_type == "siglip2":
        try:
            from transformers import Siglip2VisionModel
            vision_model = Siglip2VisionModel.from_pretrained(model_id)
        except Exception:
            try:
                from transformers import SiglipVisionModel
                vision_model = SiglipVisionModel.from_pretrained(model_id)
            except Exception:
                from transformers import AutoModel
                full = AutoModel.from_pretrained(model_id)
                vision_model = full.vision_model
                del full

        hidden_size = vision_model.config.hidden_size
        normalize_fn = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

        def extract_fn(model, pixel_values):
            return model(pixel_values=pixel_values).pooler_output

        return vision_model, hidden_size, normalize_fn, extract_fn

    elif model_type == "timm":
        import timm
        vision_model = timm.create_model(model_id, pretrained=False, num_classes=0)
        hidden_size = vision_model.num_features
        data_cfg = timm.data.resolve_data_config(model=vision_model)
        normalize_fn = transforms.Normalize(mean=data_cfg["mean"], std=data_cfg["std"])

        def extract_fn(model, pixel_values):
            return model(pixel_values)

        return vision_model, hidden_size, normalize_fn, extract_fn


def find_checkpoint(model_name):
    """Search for checkpoint across all model directories."""
    for d in MODEL_DIRS:
        path = os.path.join(d, f"{model_name}_best.pt")
        if os.path.exists(path):
            return path
    return None


def load_checkpoint(model_name, config):
    """Load trained checkpoint. Returns model, classifier, normalize, extract, meta."""
    ckpt_path = find_checkpoint(model_name)
    if ckpt_path is None:
        raise FileNotFoundError(f"Checkpoint not found for {model_name} in: {MODEL_DIRS}")

    vision_model, hidden_size, normalize_fn, extract_fn = load_backbone(config)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if "hidden_size" in ckpt:
        hidden_size = ckpt["hidden_size"]

    vision_model.load_state_dict(ckpt["vision_model"])
    classifier = ClassifierHead(hidden_size, NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])

    meta = {
        "best_f1_train": ckpt.get("best_f1", None),
        "best_epoch": ckpt.get("epoch", None),
        "label_smoothing": ckpt.get("label_smoothing", "unknown"),
        "class_thresholds": ckpt.get("class_thresholds", None),
    }

    return vision_model, classifier, normalize_fn, extract_fn, meta


# ==============================================================
# EVALUATION
# ==============================================================

@torch.no_grad()
def evaluate_model(vision_model, classifier, extract_fn, dataloader):
    """Run inference and return predictions, ground truths, confidences, indices."""
    vision_model.eval()
    classifier.eval()

    all_preds, all_gts, all_confs, all_indices = [], [], [], []

    for batch_pixels, batch_labels, batch_idx in tqdm(dataloader, desc="  Evaluating"):
        batch_pixels = batch_pixels.to(DEVICE)
        features = extract_fn(vision_model, batch_pixels)
        logits = classifier(features)
        probs = torch.softmax(logits, dim=1)
        confs, preds = probs.max(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_gts.extend(batch_labels.numpy())
        all_confs.extend(confs.cpu().numpy())
        all_indices.extend(batch_idx.numpy())

    return (
        np.array(all_preds),
        np.array(all_gts),
        np.array(all_confs),
        np.array(all_indices),
    )


# ==============================================================
# ANALYSIS HELPERS
# ==============================================================

def compute_metrics(preds, gts, class_names):
    f1_macro = f1_score(gts, preds, average="macro")
    f1_weighted = f1_score(gts, preds, average="weighted")
    accuracy = accuracy_score(gts, preds)
    precision = precision_score(gts, preds, average="macro", zero_division=0)
    recall = recall_score(gts, preds, average="macro", zero_division=0)
    per_class_f1 = f1_score(gts, preds, average=None, zero_division=0)
    cm = confusion_matrix(gts, preds, labels=list(range(len(class_names))))

    return {
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "per_class_f1": {class_names[i]: float(per_class_f1[i])
                         for i in range(len(per_class_f1))},
        "confusion_matrix": cm.tolist(),
    }


def per_class_rejection_analysis(preds, gts, confs, class_thresholds):
    """
    Apply per-class thresholds and show what gets rejected vs accepted.
    Returns stats dict.
    """
    if class_thresholds is None:
        return None

    results = {}
    total_accepted = 0
    total_rejected = 0
    total_correct_accepted = 0
    total_correct_rejected = 0

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        threshold = class_thresholds.get(cls_name, 0.60)
        pred_mask = preds == cls_idx
        n_pred = int(pred_mask.sum())

        if n_pred == 0:
            results[cls_name] = {"threshold": threshold, "predictions": 0}
            continue

        cls_confs = confs[pred_mask]
        cls_correct = (gts[pred_mask] == cls_idx)

        accepted = cls_confs >= threshold
        rejected = ~accepted

        n_accepted = int(accepted.sum())
        n_rejected = int(rejected.sum())

        acc_correct = int(cls_correct[accepted].sum()) if n_accepted > 0 else 0
        acc_precision = acc_correct / n_accepted if n_accepted > 0 else 0
        rej_correct = int(cls_correct[rejected].sum()) if n_rejected > 0 else 0

        total_accepted += n_accepted
        total_rejected += n_rejected
        total_correct_accepted += acc_correct
        total_correct_rejected += rej_correct

        results[cls_name] = {
            "threshold": float(threshold),
            "predictions": n_pred,
            "accepted": n_accepted,
            "rejected": n_rejected,
            "accepted_precision": round(acc_precision, 4),
            "rejected_would_be_correct": rej_correct,
        }

    overall_precision = total_correct_accepted / total_accepted if total_accepted > 0 else 0
    overall_rejection_rate = total_rejected / (total_accepted + total_rejected) if (total_accepted + total_rejected) > 0 else 0

    results["_overall"] = {
        "total_accepted": total_accepted,
        "total_rejected": total_rejected,
        "rejection_rate": round(overall_rejection_rate, 4),
        "accepted_precision": round(overall_precision, 4),
        "false_rejections": total_correct_rejected,
    }

    return results


def find_misclassified(preds, gts, confs, indices, samples):
    misclassified = []
    for i in range(len(preds)):
        if preds[i] != gts[i]:
            sample_idx = indices[i]
            path, label, orig_cls = samples[sample_idx]
            misclassified.append({
                "file": os.path.basename(path),
                "folder": orig_cls,
                "true_label": IDX_TO_CLASS[gts[i]],
                "pred_label": IDX_TO_CLASS[preds[i]],
                "confidence": float(confs[i]),
            })
    misclassified.sort(key=lambda x: x["confidence"], reverse=True)
    return misclassified


def print_confusion_matrix(cm, class_names, model_name):
    header = f"{'':>12s}" + "".join(f"{c:>10s}" for c in class_names)
    print(f"\n  {model_name} Confusion Matrix:")
    print(f"  {header}")
    for i, row in enumerate(cm):
        row_str = f"  {class_names[i]:>12s}" + "".join(f"{v:>10d}" for v in row)
        print(row_str)


# ==============================================================
# EVALUATE ONE DATASET
# ==============================================================

def evaluate_dataset(dataset_name, dataset_root, models_loaded):
    """Evaluate all models on one dataset. Returns results dict."""

    print(f"\n{'='*80}")
    print(f"   DATASET: {dataset_name}")
    print(f"   Path: {dataset_root}")
    print(f"{'='*80}")

    # Collect samples
    print(f"\n  Collecting samples...")
    samples = collect_samples(dataset_root)
    print(f"\n  Total samples: {len(samples)}")

    if len(samples) == 0:
        print(f"  ❌ No samples found!")
        return None

    # Distribution
    dist = defaultdict(int)
    sub_dist = defaultdict(int)
    for _, label, orig_cls in samples:
        dist[IDX_TO_CLASS[label]] += 1
        sub_dist[orig_cls] += 1

    print(f"\n  Macro class distribution:")
    for cls_name in CLASS_TO_IDX:
        print(f"    {cls_name:12s}: {dist[cls_name]:5d}")

    class_names = list(CLASS_TO_IDX.keys())
    dataset_results = {}

    for model_name, model_bundle in models_loaded.items():
        vision_model, classifier, normalize_fn, extract_fn, meta, config = model_bundle

        print(f"\n  {'─'*60}")
        print(f"  📊 {model_name} on {dataset_name}")
        print(f"  {'─'*60}")

        # Build dataloader
        val_transform = transforms.Compose([
            transforms.Resize((config["img_size"], config["img_size"])),
            transforms.ToTensor(),
        ])
        dataset = EvalDataset(samples, val_transform, normalize_fn, APPLY_PREPROCESSING)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS,
                                pin_memory=True)

        # Run inference
        t_start = time.perf_counter()
        preds, gts, confs, indices = evaluate_model(
            vision_model, classifier, extract_fn, dataloader
        )
        t_infer = time.perf_counter() - t_start
        fps = len(samples) / t_infer

        # Metrics
        metrics = compute_metrics(preds, gts, class_names)
        misclassified = find_misclassified(preds, gts, confs, indices, samples)

        # Confidence stats
        correct_mask = preds == gts
        mean_conf_correct = float(confs[correct_mask].mean()) if correct_mask.sum() > 0 else 0
        mean_conf_wrong = float(confs[~correct_mask].mean()) if (~correct_mask).sum() > 0 else 0

        # Per-class confidence
        per_class_conf = {}
        for cls_idx, cls_name in IDX_TO_CLASS.items():
            cls_mask = gts == cls_idx
            if cls_mask.sum() > 0:
                per_class_conf[cls_name] = {
                    "mean_conf": float(confs[cls_mask].mean()),
                    "min_conf": float(confs[cls_mask].min()),
                    "accuracy": float(correct_mask[cls_mask].mean() * 100),
                }

        # Per-class threshold rejection analysis
        class_thresholds = meta.get("class_thresholds")
        rejection = per_class_rejection_analysis(preds, gts, confs, class_thresholds)

        # Print results
        print(f"\n  F1 (macro):     {metrics['f1_macro']:.4f}")
        print(f"  F1 (weighted):  {metrics['f1_weighted']:.4f}")
        print(f"  Accuracy:       {metrics['accuracy']:.4f}")
        print(f"  Precision:      {metrics['precision_macro']:.4f}")
        print(f"  Recall:         {metrics['recall_macro']:.4f}")
        print(f"  Inference:      {t_infer:.1f}s ({fps:.1f} img/s)")

        print(f"\n  Per-class F1:")
        for cls_name in class_names:
            f1_val = metrics["per_class_f1"].get(cls_name, 0)
            ci = per_class_conf.get(cls_name, {})
            print(f"    {cls_name:12s}: F1={f1_val:.4f}  "
                  f"conf={ci.get('mean_conf',0):.3f} (min={ci.get('min_conf',0):.3f})  "
                  f"acc={ci.get('accuracy',0):.1f}%")

        print_confusion_matrix(metrics["confusion_matrix"], class_names, model_name)

        print(f"\n  Confidence:")
        print(f"    Correct: {mean_conf_correct:.4f}  |  Wrong: {mean_conf_wrong:.4f}")
        print(f"    Misclassified: {len(misclassified)} ({len(misclassified)/len(samples)*100:.1f}%)")

        # Per-class threshold rejection
        if rejection:
            print(f"\n  📏 Per-class threshold rejection (from training):")
            for cls_name in class_names:
                r = rejection.get(cls_name, {})
                if r.get("predictions", 0) > 0:
                    print(f"    {cls_name:12s}: thresh={r['threshold']:.3f}  "
                          f"accepted={r['accepted']}/{r['predictions']}  "
                          f"precision={r['accepted_precision']:.3f}  "
                          f"false_rej={r['rejected_would_be_correct']}")

            ov = rejection["_overall"]
            print(f"    {'OVERALL':12s}: accepted={ov['total_accepted']}  "
                  f"rejected={ov['total_rejected']} ({ov['rejection_rate']*100:.1f}%)  "
                  f"precision={ov['accepted_precision']:.3f}  "
                  f"false_rej={ov['false_rejections']}")
        else:
            print(f"\n  ⚠️  No per-class thresholds in checkpoint (old model?)")

        # Worst mistakes
        if misclassified:
            print(f"\n  🔴 Top 15 worst mistakes:")
            for m in misclassified[:15]:
                print(f"    {m['file']:40s} | {m['folder']:15s} → "
                      f"true={m['true_label']:10s} pred={m['pred_label']:10s} "
                      f"conf={m['confidence']:.3f}")

        dataset_results[model_name] = {
            "metrics": metrics,
            "inference_time_s": float(t_infer),
            "fps": float(fps),
            "confidence": {
                "mean_correct": mean_conf_correct,
                "mean_wrong": mean_conf_wrong,
                "per_class": per_class_conf,
            },
            "class_thresholds_used": class_thresholds,
            "rejection_analysis": rejection,
            "misclassified_count": len(misclassified),
            "misclassified_top20": misclassified[:20],
        }

    return {
        "dataset_name": dataset_name,
        "dataset_root": dataset_root,
        "total_samples": len(samples),
        "class_distribution": dict(dist),
        "subclass_distribution": dict(sub_dist),
        "models": dataset_results,
    }


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 80)
    print("   HELD-OUT EVALUATION — 3 Models × 2 Datasets")
    print("=" * 80)
    print(f"Device: {DEVICE}")
    print(f"Datasets: {list(EVAL_DATASETS.keys())}")
    print(f"Models: {list(MODEL_CONFIGS.keys())}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ----------------------------------------------------------
    # 1. Load all models ONCE (reuse across datasets)
    # ----------------------------------------------------------
    print(f"\n{'='*80}")
    print("   LOADING MODELS")
    print(f"{'='*80}")

    models_loaded = {}
    for model_name, config in MODEL_CONFIGS.items():
        ckpt_path = find_checkpoint(model_name)
        if ckpt_path is None:
            print(f"\n  ⚠️  SKIPPING {model_name} — checkpoint not found")
            continue

        print(f"\n  Loading {model_name} from {ckpt_path}...")
        try:
            vision_model, classifier, normalize_fn, extract_fn, meta = \
                load_checkpoint(model_name, config)
            vision_model = vision_model.to(DEVICE)
            classifier = classifier.to(DEVICE)

            print(f"    Train F1: {meta['best_f1_train']}, Epoch: {meta['best_epoch']}, "
                  f"Label smoothing: {meta['label_smoothing']}")
            if meta['class_thresholds']:
                print(f"    Per-class thresholds: {meta['class_thresholds']}")
            else:
                print(f"    ⚠️  No per-class thresholds in checkpoint")

            models_loaded[model_name] = (
                vision_model, classifier, normalize_fn, extract_fn, meta, config
            )
        except Exception as e:
            print(f"    ❌ Failed: {e}")
            import traceback
            traceback.print_exc()

    if not models_loaded:
        print("\n❌ No models loaded!")
        return

    # ----------------------------------------------------------
    # 2. Evaluate on each dataset
    # ----------------------------------------------------------
    all_dataset_results = {}

    for ds_name, ds_path in EVAL_DATASETS.items():
        result = evaluate_dataset(ds_name, ds_path, models_loaded)
        if result:
            all_dataset_results[ds_name] = result

    # ----------------------------------------------------------
    # 3. Cross-dataset comparison table
    # ----------------------------------------------------------
    print(f"\n\n{'='*100}")
    print(f"   CROSS-DATASET COMPARISON")
    print(f"{'='*100}")

    header = (f"{'Model':24s} │ {'Dataset':36s} │ "
              f"{'F1':>6s} {'Acc':>6s} {'Err':>5s} │ "
              f"{'Upper':>7s} {'Lower':>7s} {'Undwr':>7s} {'Other':>7s} │ "
              f"{'Rej%':>5s} {'APrc':>5s}")
    print(header)
    print("─" * 100)

    for model_name in MODEL_CONFIGS:
        if model_name not in models_loaded:
            continue
        for ds_name, ds_result in all_dataset_results.items():
            if model_name not in ds_result["models"]:
                continue
            mr = ds_result["models"][model_name]
            m = mr["metrics"]
            pcf = m["per_class_f1"]

            rej_rate = ""
            a_prec = ""
            if mr.get("rejection_analysis") and "_overall" in mr["rejection_analysis"]:
                ov = mr["rejection_analysis"]["_overall"]
                rej_rate = f"{ov['rejection_rate']*100:.1f}"
                a_prec = f"{ov['accepted_precision']:.3f}"

            print(f"{model_name:24s} │ {ds_name:36s} │ "
                  f"{m['f1_macro']:>6.4f} {m['accuracy']:>6.4f} {mr['misclassified_count']:>5d} │ "
                  f"{pcf.get('Upper',0):>7.4f} {pcf.get('Lower',0):>7.4f} "
                  f"{pcf.get('Underwear',0):>7.4f} {pcf.get('Other',0):>7.4f} │ "
                  f"{rej_rate:>5s} {a_prec:>5s}")

    print("─" * 100)

    # ----------------------------------------------------------
    # 4. Generalization gap
    # ----------------------------------------------------------
    print(f"\n  Generalization gap (train F1 → held-out F1):")
    for model_name in models_loaded:
        meta = models_loaded[model_name][4]
        train_f1 = meta["best_f1_train"]
        for ds_name, ds_result in all_dataset_results.items():
            if model_name in ds_result["models"]:
                held_f1 = ds_result["models"][model_name]["metrics"]["f1_macro"]
                gap = (train_f1 - held_f1) if train_f1 else None
                if gap is not None:
                    ind = "🟢" if gap < 0.03 else ("🟡" if gap < 0.06 else "🔴")
                    print(f"    {ind} {model_name:24s} on {ds_name:30s}: "
                          f"{train_f1:.4f} → {held_f1:.4f}  (gap={gap:+.4f})")

    # ----------------------------------------------------------
    # 5. Save results
    # ----------------------------------------------------------
    results_path = os.path.join(OUTPUT_DIR, "eval_all_results.json")

    save_data = {}
    for ds_name, ds_result in all_dataset_results.items():
        save_data[ds_name] = ds_result.copy()

    with open(results_path, "w") as f:
        json.dump(save_data, f, indent=2, default=str)
    print(f"\n✅ All results saved to: {results_path}")

    # Save misclassified per model per dataset
    misc_path = os.path.join(OUTPUT_DIR, "misclassified_all.json")
    misc_data = {}
    for ds_name, ds_result in all_dataset_results.items():
        misc_data[ds_name] = {}
        for model_name, mr in ds_result["models"].items():
            misc_data[ds_name][model_name] = mr.get("misclassified_top20", [])
    with open(misc_path, "w") as f:
        json.dump(misc_data, f, indent=2)
    print(f"✅ Misclassified details saved to: {misc_path}")

    # Cleanup
    for model_name, bundle in models_loaded.items():
        del bundle
    torch.cuda.empty_cache()
    gc.collect()


if __name__ == "__main__":
    main()

### Temperature scaling

In [ ]:
"""
============================================================
   TEMPERATURE SCALING — DINOv3 + SigLIP2
   Finds optimal T per model on validation set.
   No retraining — just calibration post-hoc.
============================================================

Run in Colab:
    !pip install timm transformers --quiet
    Then run this cell.

After running, update production config JSONs with the
temperature values and use: probs = softmax(logits / T)
"""

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize_scalar
from tqdm import tqdm
import timm
import json

# ==============================================================
# CONFIG
# ==============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DINOV3_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/DINOv3_ViT_Base_best.pt"
SIGLIP2_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/SigLIP2_SO400M_best.pt"

DINOV3_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/dinov3_production_config.json"
SIGLIP2_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/siglip2_production_config.json"

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data",
]

IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_CLASSES = 4

BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING
# ==============================================================

UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

# ==============================================================
# PREPROCESSING (exact training match)
# ==============================================================

def preprocess_crop(img_np):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > BLACK_THRESH, 255, 0).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)
    gray_bg = np.full_like(img_np, BACKGROUND_COLOR, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)
    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
        from PIL import Image
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))
    mx, my = int(bw * MARGIN_RATIO), int(bh * MARGIN_RATIO)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]
    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    from PIL import Image
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))


class ValDataset(Dataset):
    def __init__(self, samples, transform, normalize):
        self.samples = samples
        self.transform = transform
        self.normalize = normalize

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            from PIL import Image
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        else:
            img = preprocess_crop(img_bgr)
        px = self.transform(img)
        px = self.normalize(px)
        return px, label


# ==============================================================
# CLASSIFIER HEAD
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))


# ==============================================================
# COLLECT LOGITS FROM MODEL
# ==============================================================

def collect_logits(vision_model, classifier, loader):
    """Run inference, collect raw LOGITS (before softmax) and true labels."""
    all_logits = []
    all_labels = []

    vision_model.eval()
    classifier.eval()

    with torch.no_grad(), torch.amp.autocast("cuda"):
        for px, labels in tqdm(loader, desc="  Collecting logits"):
            px = px.to(DEVICE)
            feats = vision_model(px)

            # Handle HuggingFace models that return objects
            if hasattr(feats, 'pooler_output'):
                feats = feats.pooler_output
            elif hasattr(feats, 'last_hidden_state'):
                feats = feats.last_hidden_state[:, 0]

            logits = classifier(feats)  # RAW logits, no softmax
            all_logits.append(logits.cpu())
            all_labels.append(labels)

    return torch.cat(all_logits), torch.cat(all_labels)


# ==============================================================
# TEMPERATURE SCALING
# ==============================================================

def find_optimal_temperature(logits, labels, t_range=(0.5, 10.0)):
    """
    Find T that minimizes Negative Log Likelihood on calibration set.
    NLL = -mean(log(softmax(logits/T)[correct_class]))
    """
    def nll_at_t(T):
        scaled = logits / T
        log_probs = F.log_softmax(scaled, dim=1)
        nll = F.nll_loss(log_probs, labels)
        return nll.item()

    result = minimize_scalar(nll_at_t, bounds=t_range, method='bounded')
    return result.x, result.fun


def compute_ece(probs, labels, n_bins=15):
    """Expected Calibration Error."""
    confidences = probs.max(dim=1).values
    predictions = probs.argmax(dim=1)
    accuracies = predictions.eq(labels)

    ece = 0.0
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].float().mean()
        bin_conf = confidences[mask].mean()
        bin_size = mask.sum().float() / len(labels)
        ece += (bin_acc - bin_conf).abs() * bin_size

    return ece.item()


def analyze_calibration(logits, labels, T, model_name):
    """Show detailed before/after calibration stats."""

    print(f"\n  {'─'*55}")
    print(f"  {model_name} — Temperature = {T:.3f}")
    print(f"  {'─'*55}")

    # Before (T=1)
    probs_before = F.softmax(logits, dim=1)
    preds_before = probs_before.argmax(dim=1)
    confs_before = probs_before.max(dim=1).values
    correct_mask = preds_before.eq(labels)

    # After (T=optimal)
    probs_after = F.softmax(logits / T, dim=1)
    preds_after = probs_after.argmax(dim=1)
    confs_after = probs_after.max(dim=1).values

    # Accuracy unchanged (T doesn't change argmax)
    acc = correct_mask.float().mean().item()
    print(f"  Accuracy: {acc:.4f} (unchanged by temperature)")

    # ECE
    ece_before = compute_ece(probs_before, labels)
    ece_after = compute_ece(probs_after, labels)
    print(f"\n  ECE (Expected Calibration Error):")
    print(f"    Before (T=1):   {ece_before:.4f}")
    print(f"    After  (T={T:.2f}): {ece_after:.4f}  ({'✅ improved' if ece_after < ece_before else '⚠️ worse'})")

    # Confidence distribution
    print(f"\n  Confidence Distribution:")
    print(f"    {'':15s} {'Before (T=1)':>15s} {'After (T='+f'{T:.2f}'+')':>15s}")

    c_correct_before = confs_before[correct_mask]
    c_wrong_before = confs_before[~correct_mask]
    c_correct_after = confs_after[correct_mask]
    c_wrong_after = confs_after[~correct_mask]

    print(f"    {'Correct mean':15s} {c_correct_before.mean():.3f}           {c_correct_after.mean():.3f}")
    print(f"    {'Wrong mean':15s} {c_wrong_before.mean():.3f}           {c_wrong_after.mean():.3f}")
    print(f"    {'Gap':15s} {(c_correct_before.mean()-c_wrong_before.mean()):.3f}           {(c_correct_after.mean()-c_wrong_after.mean()):.3f}")

    # Confidence percentiles
    print(f"\n  Confidence Percentiles (ALL predictions):")
    for p in [10, 25, 50, 75, 90]:
        # Line 265-266: cast to float
        b = torch.quantile(confs_before.float(), p/100).item()
        a = torch.quantile(confs_after.float(), p/100).item()
        print(f"    P{p:02d}: {b:.3f} → {a:.3f}")

    # Per-class analysis
    print(f"\n  Per-Class Confidence (After T={T:.2f}):")
    for cls_idx, cls_name in IDX_TO_CLASS.items():
        cls_mask = labels == cls_idx
        if cls_mask.sum() == 0:
            continue
        pred_mask = preds_after == cls_idx
        cls_correct = (preds_after[cls_mask] == cls_idx)
        cls_confs = confs_after[cls_mask]
        correct_confs = cls_confs[cls_correct]
        wrong_confs = cls_confs[~cls_correct]

        print(f"    {cls_name:12s}: n={cls_mask.sum():4d}  "
              f"acc={cls_correct.float().mean():.3f}  "
              f"conf_correct={correct_confs.mean():.3f}  "
              + (f"conf_wrong={wrong_confs.mean():.3f}" if len(wrong_confs) > 0 else "no errors"))

    # Threshold effectiveness after scaling
    print(f"\n  Threshold Analysis (After T={T:.2f}):")
    for thresh in [0.50, 0.60, 0.70, 0.80, 0.90]:
        accepted = confs_after >= thresh
        n_accepted = accepted.sum().item()
        if n_accepted == 0:
            continue
        acc_accepted = correct_mask[accepted].float().mean().item()
        reject_pct = (1 - n_accepted / len(labels)) * 100
        print(f"    thresh={thresh:.2f}: accept={n_accepted:4d} ({100-reject_pct:.1f}%)  "
              f"precision={acc_accepted:.4f}  reject={reject_pct:.1f}%")

    return T


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print("  TEMPERATURE SCALING — DINOv3 + SigLIP2")
    print("  (No retraining — just calibration)")
    print("=" * 60)

    # ── Collect val samples (same split as training) ──
    print("\nCollecting samples...")
    all_samples = []
    for root in TRAIN_ROOTS:
        if not os.path.isdir(root):
            continue
        for cls_name in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path):
                continue
            label = map_class(cls_name)
            if label is None:
                continue
            for f in os.listdir(cls_path):
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    all_samples.append((os.path.join(cls_path, f), label))

    print(f"  Total samples: {len(all_samples)}")
    labels = [l for _, l in all_samples]
    _, val_samples = train_test_split(all_samples, test_size=0.2, stratify=labels, random_state=42)
    print(f"  Val split: {len(val_samples)} (same random_state=42 as training)")

    results = {}

    # ==============================================================
    # DINOv3
    # ==============================================================
    if os.path.exists(DINOV3_CHECKPOINT):
        print(f"\n{'='*60}")
        print(f"  DINOv3_ViT_Base")
        print(f"{'='*60}")

        ckpt = torch.load(DINOV3_CHECKPOINT, map_location=DEVICE, weights_only=False)
        hidden_size = ckpt["hidden_size"]
        print(f"  Loaded: F1={ckpt['best_f1']:.4f}, epoch={ckpt['epoch']}")

        vision_model = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=False, num_classes=0)
        vision_model.load_state_dict(ckpt["vision_model"])
        vision_model = vision_model.to(DEVICE).eval()

        classifier = ClassifierHead(hidden_size, NUM_CLASSES)
        classifier.load_state_dict(ckpt["classifier"])
        classifier = classifier.to(DEVICE).eval()

        # Normalization from timm
        _cfg = timm.data.resolve_data_config(model=vision_model)
        normalize = transforms.Normalize(mean=_cfg["mean"], std=_cfg["std"])
        transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

        loader = DataLoader(ValDataset(val_samples, transform, normalize),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        print(f"  Collecting logits on {len(val_samples)} val samples...")
        logits, labels_t = collect_logits(vision_model, classifier, loader)
        print(f"  Logits shape: {logits.shape}")

        # Find optimal T
        print(f"\n  Optimizing temperature...")
        T_opt, nll_opt = find_optimal_temperature(logits, labels_t)
        print(f"  Optimal T = {T_opt:.4f} (NLL = {nll_opt:.4f})")

        analyze_calibration(logits, labels_t, T_opt, "DINOv3")
        results["DINOv3"] = T_opt

        # Cleanup GPU
        del vision_model, classifier
        torch.cuda.empty_cache()

    # ==============================================================
    # SigLIP2
    # ==============================================================
    if os.path.exists(SIGLIP2_CHECKPOINT):
        print(f"\n{'='*60}")
        print(f"  SigLIP2_SO400M")
        print(f"{'='*60}")

        ckpt = torch.load(SIGLIP2_CHECKPOINT, map_location=DEVICE, weights_only=False)
        hidden_size = ckpt["hidden_size"]
        print(f"  Loaded: F1={ckpt['best_f1']:.4f}, epoch={ckpt['epoch']}")

        # Load SigLIP2 backbone
        try:
            from transformers import Siglip2VisionModel
            vision_model = Siglip2VisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
        except Exception:
            try:
                from transformers import SiglipVisionModel
                vision_model = SiglipVisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
            except Exception:
                from transformers import AutoModel
                full = AutoModel.from_pretrained("google/siglip2-so400m-patch14-384")
                vision_model = full.vision_model
                del full

        vision_model.load_state_dict(ckpt["vision_model"])
        vision_model = vision_model.to(DEVICE).eval()

        classifier = ClassifierHead(hidden_size, NUM_CLASSES)
        classifier.load_state_dict(ckpt["classifier"])
        classifier = classifier.to(DEVICE).eval()

        # SigLIP2 normalization: mean=0.5, std=0.5
        normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

        loader = DataLoader(ValDataset(val_samples, transform, normalize),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        print(f"  Collecting logits on {len(val_samples)} val samples...")
        logits, labels_t = collect_logits(vision_model, classifier, loader)
        print(f"  Logits shape: {logits.shape}")

        print(f"\n  Optimizing temperature...")
        T_opt, nll_opt = find_optimal_temperature(logits, labels_t)
        print(f"  Optimal T = {T_opt:.4f} (NLL = {nll_opt:.4f})")

        analyze_calibration(logits, labels_t, T_opt, "SigLIP2")
        results["SigLIP2"] = T_opt

        del vision_model, classifier
        torch.cuda.empty_cache()

    # ==============================================================
    # SAVE TO CONFIG
    # ==============================================================

    print(f"\n\n{'='*60}")
    print(f"  RESULTS SUMMARY")
    print(f"{'='*60}")

    for model_name, T in results.items():
        print(f"  {model_name}: T = {T:.4f}")

    # Update DINOv3 config
    if "DINOv3" in results and os.path.exists(DINOV3_CONFIG):
        with open(DINOV3_CONFIG, "r") as f:
            cfg = json.load(f)
        cfg["temperature"] = round(results["DINOv3"], 4)
        with open(DINOV3_CONFIG, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"\n  ✅ Updated {DINOV3_CONFIG} with temperature={cfg['temperature']}")

    # Update SigLIP2 config
    if "SigLIP2" in results and os.path.exists(SIGLIP2_CONFIG):
        with open(SIGLIP2_CONFIG, "r") as f:
            cfg = json.load(f)
        cfg["temperature"] = round(results["SigLIP2"], 4)
        with open(SIGLIP2_CONFIG, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"  ✅ Updated {SIGLIP2_CONFIG} with temperature={cfg['temperature']}")

    print(f"\n  Production usage:")
    print(f"  ─────────────────")
    print(f"  config = json.load(open('dinov3_production_config.json'))")
    print(f"  T = config['temperature']")
    print(f"  # After TRT inference gives raw logits:")
    print(f"  scaled = logits / T")
    print(f"  probs = softmax(scaled)")
    print(f"  # Now confidence values are properly calibrated!")

    print(f"\n{'='*60}")
    print(f"  DONE")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

### Per-class thresholds (run after temperature scaling)

In [ ]:
"""
Recompute per-class thresholds WITH temperature scaling.
Run AFTER temperature_scaling.py has saved T values to configs.
"""
import os, json, cv2, torch, timm
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DEVICE = "cuda"
IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_CLASSES = 4
TARGET_PRECISION = 0.95  # Same as before

BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

DINOV3_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/DINOv3_ViT_Base_best.pt"
SIGLIP2_CHECKPOINT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/SigLIP2_SO400M_best.pt"
DINOV3_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/dinov3_production_config.json"
SIGLIP2_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models/siglip2_production_config.json"

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data",
]

UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]
CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

def preprocess_crop(img_np):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > BLACK_THRESH, 255, 0).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)
    gray_bg = np.full_like(img_np, BACKGROUND_COLOR, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)
    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        from PIL import Image
        return Image.fromarray(np.full((IMG_SIZE, IMG_SIZE, 3), 128, dtype=np.uint8))
    mx, my = int(bw * MARGIN_RATIO), int(bh * MARGIN_RATIO)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]
    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    from PIL import Image
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

class ValDataset(Dataset):
    def __init__(self, samples, transform, normalize):
        self.samples = samples
        self.transform = transform
        self.normalize = normalize
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            from PIL import Image
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        else:
            img = preprocess_crop(img_bgr)
        return self.normalize(self.transform(img)), label

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x): return self.fc(self.dropout(x))


def collect_logits(vision_model, classifier, loader):
    all_logits, all_labels = [], []
    vision_model.eval(); classifier.eval()
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for px, labels in tqdm(loader, desc="  Collecting logits"):
            feats = vision_model(px.to(DEVICE))
            if hasattr(feats, 'pooler_output'): feats = feats.pooler_output
            elif hasattr(feats, 'last_hidden_state'): feats = feats.last_hidden_state[:, 0]
            all_logits.append(classifier(feats).cpu())
            all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)


def compute_thresholds_with_temperature(logits, labels, T, target_precision=0.95):
    """Compute per-class thresholds on temperature-scaled probabilities."""
    probs = F.softmax(logits / T, dim=1)
    preds = probs.argmax(dim=1)
    confs = probs.max(dim=1).values

    print(f"\n  Computing thresholds (T={T:.4f}, target_precision={target_precision})")
    thresholds = {}

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        # Get all samples predicted as this class
        pred_mask = preds == cls_idx
        if pred_mask.sum() == 0:
            thresholds[cls_name.lower()] = 0.5
            continue

        pred_confs = confs[pred_mask].float()
        pred_labels = labels[pred_mask]
        pred_correct = (pred_labels == cls_idx)

        # Sort by confidence descending
        sorted_idx = torch.argsort(pred_confs, descending=True)
        pred_confs = pred_confs[sorted_idx]
        pred_correct = pred_correct[sorted_idx]

        # Find threshold where cumulative precision >= target
        cum_correct = torch.cumsum(pred_correct.float(), dim=0)
        cum_total = torch.arange(1, len(pred_correct) + 1, dtype=torch.float32)
        cum_precision = cum_correct / cum_total

        # Find the lowest confidence where precision is still >= target
        valid = cum_precision >= target_precision
        if valid.any():
            last_valid_idx = valid.nonzero()[-1].item()
            thresh = float(pred_confs[last_valid_idx])
        else:
            # Can't achieve target precision, use max confidence as threshold
            thresh = float(pred_confs[0]) + 0.01

        # Stats
        n_above = (pred_confs >= thresh).sum().item()
        if n_above > 0:
            actual_prec = float(pred_correct[pred_confs >= thresh].float().mean())
        else:
            actual_prec = 0.0

        conf_correct = float(pred_confs[pred_correct].mean()) if pred_correct.any() else 0
        conf_wrong = float(pred_confs[~pred_correct].mean()) if (~pred_correct).any() else 0

        thresholds[cls_name.lower()] = round(thresh, 4)

        print(f"    {cls_name:12s}: thresh={thresh:.4f}  "
              f"accept={n_above}/{pred_mask.sum()}  "
              f"precision={actual_prec:.3f}  "
              f"conf_correct={conf_correct:.3f}  "
              f"conf_wrong={conf_wrong:.3f}")

    return thresholds


def process_model(name, ckpt_path, config_path, load_vision_fn, get_normalize_fn):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    # Load config to get T
    with open(config_path) as f:
        cfg = json.load(f)
    T = cfg.get("temperature", 1.0)
    print(f"  Temperature: {T}")

    # Load model
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    vision_model = load_vision_fn(ckpt)
    classifier = ClassifierHead(ckpt["hidden_size"], NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])
    classifier = classifier.to(DEVICE).eval()
    normalize = get_normalize_fn(vision_model)
    transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

    loader = DataLoader(ValDataset(val_samples, transform, normalize),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    logits, labels_t = collect_logits(vision_model, classifier, loader)
    thresholds = compute_thresholds_with_temperature(logits, labels_t, T, TARGET_PRECISION)

    # Update config
    cfg["class_thresholds"] = thresholds
    with open(config_path, "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"\n  ✅ Saved updated thresholds to {config_path}")
    print(f"     {thresholds}")

    del vision_model, classifier
    torch.cuda.empty_cache()
    return thresholds


# ── Collect samples ──
print("Collecting samples...")
all_samples = []
for root in TRAIN_ROOTS:
    if not os.path.isdir(root): continue
    for cls_name in sorted(os.listdir(root)):
        cls_path = os.path.join(root, cls_name)
        if not os.path.isdir(cls_path): continue
        label = map_class(cls_name)
        if label is None: continue
        for f in os.listdir(cls_path):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                all_samples.append((os.path.join(cls_path, f), label))

labels = [l for _, l in all_samples]
_, val_samples = train_test_split(all_samples, test_size=0.2, stratify=labels, random_state=42)
print(f"  Val: {len(val_samples)} samples")


# ── DINOv3 ──
def load_dino(ckpt):
    vm = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=False, num_classes=0)
    vm.load_state_dict(ckpt["vision_model"])
    return vm.to(DEVICE).eval()

def dino_normalize(vm):
    _cfg = timm.data.resolve_data_config(model=vm)
    return transforms.Normalize(mean=_cfg["mean"], std=_cfg["std"])

dino_thresh = process_model("DINOv3", DINOV3_CHECKPOINT, DINOV3_CONFIG, load_dino, dino_normalize)


# ── SigLIP2 ──
def load_siglip(ckpt):
    try:
        from transformers import Siglip2VisionModel
        vm = Siglip2VisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
    except:
        try:
            from transformers import SiglipVisionModel
            vm = SiglipVisionModel.from_pretrained("google/siglip2-so400m-patch14-384")
        except:
            from transformers import AutoModel
            full = AutoModel.from_pretrained("google/siglip2-so400m-patch14-384")
            vm = full.vision_model; del full
    vm.load_state_dict(ckpt["vision_model"])
    return vm.to(DEVICE).eval()

def siglip_normalize(vm):
    return transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

siglip_thresh = process_model("SigLIP2", SIGLIP2_CHECKPOINT, SIGLIP2_CONFIG, load_siglip, siglip_normalize)


print(f"\n{'='*60}")
print(f"  DONE — New thresholds (post temperature scaling)")
print(f"{'='*60}")
print(f"  DINOv3  (T=1.83): {dino_thresh}")
print(f"  SigLIP2 (T=2.67): {siglip_thresh}")
print(f"\n  Copy updated JSON configs to Orin and restart.")